# Tokenizer training


In [ ]:
import json
import collections
from pathlib import Path

In [ ]:
MANIFEST_DIR = Path("data/manifests")
VOCAB_SIZE = 50000
SPECIAL_BASE = [
    "<|endoftext|>",
    "<|startoftranscript|>",
]
LANGUAGES = [
    "en", "zh", "de", "es", "ru", "ko", "fr", "ja", "pt", "tr", "pl", "ca",
    "nl", "ar", "sv", "it", "id", "hi", "fi", "vi", "he", "uk", "el", "ms",
    "cs", "ro", "da", "hu", "ta", "no", "th", "ur", "hr", "bg", "lt", "la",
    "mi", "ml", "cy", "sk", "te", "fa", "lv", "bn", "sr", "az", "sl", "kn",
    "et", "mk", "br", "eu", "is", "hy", "ne", "mn", "bs", "kk", "sq", "sw",
    "gl", "mr", "pa", "si", "km", "sn", "yo", "so", "af", "oc", "ka", "be",
    "tg", "sd", "gu", "am", "yi", "lo", "uz", "fo", "ht", "ps", "tk", "nn",
    "mt", "sa", "lb", "my", "bo", "tl", "mg", "as", "tt", "haw", "ln", "ha",
    "ba", "jw", "su",
]
TASK_TOKENS = ["<|translate|>", "<|transcribe|>"]
CONTROL_TOKENS = ["<|startoflm|>", "<|startofprev|>", "<|nospeech|>", "<|notimestamps|>"]
N_TIMESTAMPS = 1501

In [ ]:
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\u00a1"), ord("\u00ac") + 1)) + list(range(ord("\u00ae"), ord("\u00ff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

BYTE_ENCODER = bytes_to_unicode()
BYTE_DECODER = {v: k for k, v in BYTE_ENCODER.items()}

def to_byte_units(text):
    return "".join(BYTE_ENCODER[b] for b in text.encode("utf-8"))

def from_byte_units(units):
    data = bytes(BYTE_DECODER[u] for u in units)
    return data.decode("utf-8", errors="replace")

In [ ]:
import regex

PRETOKEN_PATTERN = regex.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

def pretokenize(text):
    return PRETOKEN_PATTERN.findall(text)

def word_frequencies(manifests):
    freqs = collections.Counter()
    for path in manifests:
        with open(path) as f:
            for line in f:
                row = json.loads(line)
                for piece in pretokenize(row["text"]):
                    freqs[to_byte_units(piece)] += 1
                if row.get("translation"):
                    for piece in pretokenize(row["translation"]):
                        freqs[to_byte_units(piece)] += 1
    return freqs

manifest_paths = sorted(MANIFEST_DIR.glob("*.jsonl"))
freqs = word_frequencies(manifest_paths)
print(len(freqs))

In [ ]:
def train_bpe(freqs, vocab_size):
    words = {tuple(w): c for w, c in freqs.items()}
    vocab = {chr_: None for w in words for chr_ in w}
    merges = []
    base = sorted(vocab)
    while len(base) + len(merges) < vocab_size:
        pair_counts = collections.Counter()
        for word, count in words.items():
            for a, b in zip(word, word[1:]):
                pair_counts[(a, b)] += count
        if not pair_counts:
            break
        (a, b), best = pair_counts.most_common(1)[0]
        if best < 2:
            break
        merges.append((a, b))
        merged = a + b
        new_words = {}
        for word, count in words.items():
            out = []
            i = 0
            while i < len(word):
                if i + 1 < len(word) and word[i] == a and word[i + 1] == b:
                    out.append(merged)
                    i += 2
                else:
                    out.append(word[i])
                    i += 1
            new_words[tuple(out)] = new_words.get(tuple(out), 0) + count
        words = new_words
        if len(merges) % 500 == 0:
            print(len(base) + len(merges))
    return base, merges

base_alphabet, merges = train_bpe(freqs, VOCAB_SIZE)
print(len(base_alphabet), len(merges))

In [ ]:
class BPETokenizer:
    def __init__(self, base_alphabet, merges):
        self.vocab = {}
        for unit in base_alphabet:
            self.vocab[unit] = len(self.vocab)
        for a, b in merges:
            self.vocab[a + b] = len(self.vocab)
        self.ranks = {pair: i for i, pair in enumerate(merges)}
        self.id_to_token = {i: t for t, i in self.vocab.items()}
        self.cache = {}

    def bpe(self, units):
        if units in self.cache:
            return self.cache[units]
        word = list(units)
        while len(word) > 1:
            pairs = [(self.ranks.get((a, b), float("inf")), i) for i, (a, b) in enumerate(zip(word, word[1:]))]
            rank, idx = min(pairs)
            if rank == float("inf"):
                break
            word[idx:idx + 2] = [word[idx] + word[idx + 1]]
        self.cache[units] = word
        return word

    def encode(self, text):
        ids = []
        for piece in pretokenize(text):
            for token in self.bpe(to_byte_units(piece)):
                ids.append(self.vocab[token])
        return ids

    def decode(self, ids):
        units = "".join(self.id_to_token[i] for i in ids if i in self.id_to_token)
        return from_byte_units(units)

tok = BPETokenizer(base_alphabet, merges)
sample = "The quarterly revenue grew 14% to $2.3M, señor."
ids = tok.encode(sample)
print(len(ids), tok.decode(ids) == sample)

In [ ]:
class MultitaskTokenizer:
    def __init__(self, bpe):
        self.bpe = bpe
        n = len(bpe.vocab)
        self.eot = n
        self.sot = n + 1
        self.language_offset = n + 2
        self.language_ids = {lang: self.language_offset + i for i, lang in enumerate(LANGUAGES)}
        after_langs = self.language_offset + len(LANGUAGES)
        self.translate = after_langs
        self.transcribe = after_langs + 1
        self.start_of_lm = after_langs + 2
        self.start_of_prev = after_langs + 3
        self.no_speech = after_langs + 4
        self.no_timestamps = after_langs + 5
        self.timestamp_begin = after_langs + 6
        self.n_vocab = self.timestamp_begin + N_TIMESTAMPS

    def timestamp_token(self, seconds):
        return self.timestamp_begin + int(round(seconds / 0.02))

    def timestamp_seconds(self, token):
        return (token - self.timestamp_begin) * 0.02

    def sot_sequence(self, language, task, timestamps=True):
        seq = [self.sot, self.language_ids[language]]
        seq.append(self.translate if task == "translate" else self.transcribe)
        if not timestamps:
            seq.append(self.no_timestamps)
        return seq

    def encode_target(self, row, timestamps=None):
        text = row["translation"] if row["task"] == "translate" and row.get("translation") else row["text"]
        seq = self.sot_sequence(row["language"], row["task"], timestamps=timestamps is not None)
        if timestamps is not None:
            for start, end, segment_text in timestamps:
                seq.append(self.timestamp_token(start))
                seq.extend(self.bpe.encode(segment_text))
                seq.append(self.timestamp_token(end))
        else:
            seq.extend(self.bpe.encode(text))
        seq.append(self.eot)
        return seq

mt = MultitaskTokenizer(tok)
row = {"language": "es", "task": "translate", "text": "hola equipo", "translation": "hello team"}
print(mt.encode_target(row))
print(mt.n_vocab)

In [ ]:
def special_token_table(mt):
    table = {
        "<|endoftext|>": mt.eot,
        "<|startoftranscript|>": mt.sot,
        "<|translate|>": mt.translate,
        "<|transcribe|>": mt.transcribe,
        "<|startoflm|>": mt.start_of_lm,
        "<|startofprev|>": mt.start_of_prev,
        "<|nospeech|>": mt.no_speech,
        "<|notimestamps|>": mt.no_timestamps,
    }
    for lang, idx in mt.language_ids.items():
        table[f"<|{lang}|>"] = idx
    for i in range(N_TIMESTAMPS):
        table[f"<|{i * 0.02:.2f}|>"] = mt.timestamp_begin + i
    return table

def export_tokenizer(mt, path):
    added = [
        {"id": idx, "content": content, "special": True}
        for content, idx in sorted(special_token_table(mt).items(), key=lambda kv: kv[1])
    ]
    payload = {
        "model": {
            "type": "BPE",
            "vocab": mt.bpe.vocab,
            "merges": [f"{a} {b}" for a, b in merges],
        },
        "added_tokens": added,
    }
    with open(path, "w") as f:
        json.dump(payload, f, ensure_ascii=False)

export_tokenizer(mt, "assets/tokenizer.json")
print(Path("assets/tokenizer.json").stat().st_size)

In [ ]:
def roundtrip_check(mt, samples):
    failures = []
    for s in samples:
        ids = mt.bpe.encode(s)
        if mt.bpe.decode(ids) != s:
            failures.append(s)
    return failures

samples = [
    "hello world",
    "naïve café — résumé",
    "数字 と 漢字",
    "$1,234.56 at 3:45pm",
    "don't-stop believing!!",
    "переговоры завтра в 10:00",
]
print(roundtrip_check(mt, samples))

In [ ]:
def grammar_check(mt, seq):
    if seq[0] != mt.sot:
        return "missing sot"
    if not (mt.language_offset <= seq[1] < mt.language_offset + len(LANGUAGES)):
        return "missing language"
    if seq[2] not in (mt.translate, mt.transcribe):
        return "missing task"
    if seq[-1] != mt.eot:
        return "missing eot"
    stamps = [t for t in seq if t >= mt.timestamp_begin]
    if len(stamps) % 2 != 0:
        return "unpaired timestamps"
    for a, b in zip(stamps, stamps[1:]):
        if b < a:
            return "decreasing timestamps"
    return "ok"

seq = mt.encode_target(
    {"language": "en", "task": "transcribe", "text": "one two three"},
    timestamps=[(0.0, 1.28, "one two"), (1.28, 2.5, "three")],
)
print(grammar_check(mt, seq))